#### Importing all modules and classes

In [1]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv
from langchain.agents import create_agent

# load env
load_dotenv()

True

##### Define a tool for checking weather

In [2]:
def check_weather(location: str)->str:
    """ Get the current weather in a given location."""
    # Get the OpenWeatherMap API key from the environment variable
    api_key = os.getenv("OPENWEATHER_API_KEY")

    # OpenWeatherMap API endpoint
    url = "https://api.openweathermap.org/data/2.5/weather"

    # Query parameters
    params = {
        "q": location,
        "appid":api_key,
        "units":"metric"
    }

    # Send the GET http request
    response = requests.get(url,params=params, timeout=10)

    # Check if the request was successful
    if response.status_code != 200:
        return f"Could not get weather for {location}. Error: {response.text}"

    # Parse the JSON response
    data = response.json()
    print(data)
    city = data["name"]
    country = data["sys"]["country"]
    temperature = data["main"]["temp"]
    feels_like = data["main"]["feels_like"]
    description = data["weather"][0]["description"]
    humidity = data["main"]["humidity"]

    # Return the weather information as a string
    return (
        f"Current weather in {city}, {country}: "
        f"{description}, temperature {temperature}°C, "
        f"feels like {feels_like}°C, humidity {humidity}%."
    )


#### Creating the agent

In [3]:
# Create the agent using the OpenAI API
"""
Note: The OpenAI API key is stored in the .env file and it's auto-detected by LangChain/OpenAI.
How it works:
- Python reads the .env file with the aid of the `load_dotenv()` function.
- Variables inside it become available as environment variables.
- LangChain/OpenAI automatically checks for OPENAI_API_KEY. So we don't need to pass it explicitly.
"""


graph = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[check_weather],
    system_prompt="You are an AI Weather Assistant. Use the weather tool when asked about weather"
)

In [4]:
inputs = {
    "messages": [
        {
            "role": "user",
            "content": "Lagos, Nigeria?"
        }
    ]
}

##### Run the agent and parse the agent's output while streaming.

In [5]:
# Streaming helps us get the output in real-time as it's generated
for chunk in graph.stream(inputs, stream_mode="updates"):
    # 1. Model output
    if "model" in chunk:
        messages = chunk["model"]["messages"]

        for msg in messages:
            # Tool call request
            if msg.tool_calls:
                print("\n🤖 Agent wants to use a tool:")
                for tool_call in msg.tool_calls:
                    print(f"Tool: {tool_call['name']}")
                    print(f"Arguments: {tool_call['args']}")

            # Final AI response
            if msg.content:
                print("\n✅ Final Answer:")
                print(msg.content)

            # Token usage
            if hasattr(msg, "usage_metadata") and msg.usage_metadata:
                print("\n📊 Token Usage:")
                pprint(msg.usage_metadata)

    # 2. Tool output
    if "tools" in chunk:
        messages = chunk["tools"]["messages"]

        for msg in messages:
            print("\n🛠️ Tool Result:")
            print(f"Tool: {msg.name}")
            print(f"Output: {msg.content}")


InvalidUpdateError: Expected dict, got <bound method Kernel.raw_input of <ipykernel.ipkernel.IPythonKernel object at 0x107b66510>>
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE